In [1]:
!git clone https://github.com/HaiAu2501/EL4TF

Cloning into 'EL4TF'...
remote: Enumerating objects: 1783, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 1783 (delta 70), reused 67 (delta 60), pack-reused 1694 (from 3)
Receiving objects: 100% (1783/1783), 29.14 MiB | 23.68 MiB/s, done.
Resolving deltas: 100% (932/932), done.


In [2]:
%cd EL4TF

/content/EL4TF


### Random Fourier Features (RFF) → Logistic/SVM tuyến tính + non-ensemble

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

import numpy as np
from loaders._gen_binary import generate_data
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import balanced_accuracy_score, make_scorer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
_ = generate_data(verbose=True)


Generated dataset with 1995 samples.
Shapes: [(1596, 25), (0, 25), (399, 25)]
Label distribution: Counter({np.int64(1): 817, np.int64(0): 779})
[WARNING] If you use a tree-based model, consider setting use_scaler=False.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import balanced_accuracy_score, make_scorer
import numpy as np

# ========== PHIÊN BẢN 1: NON-ENSEMBLE (đơn giản) ==========
bacc = []

for seed in range(30):
    pack = generate_data(seed=seed, use_scaler=False)  # RFF pipeline đã có StandardScaler
    X_train, y_train = pack["train"]
    X_test, y_test = pack["test"]

    tscv = TimeSeriesSplit(n_splits=3)

    # Pipeline với RFF + Logistic Regression
    rff_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("rff", RBFSampler(random_state=42)),
        ("clf", LogisticRegression(max_iter=2000, random_state=42, n_jobs=-1))
    ])

    # Param grid cho RFF
    param_grid = {
        "rff__gamma": [0.1, 0.5, 1.0, 2.0],           # kernel bandwidth
        "rff__n_components": [500, 1000, 2000],       # số features RFF
        "clf__C": [0.1, 1.0, 10.0],                   # regularization strength
    }

    # Scorer
    scorer = make_scorer(balanced_accuracy_score)

    search = GridSearchCV(
        estimator=rff_pipeline,
        param_grid=param_grid,
        cv=tscv,
        scoring=scorer,
        n_jobs=-1,
        refit=True
    )

    search.fit(X_train, y_train)

    y_preds = search.predict(X_test)
    score = balanced_accuracy_score(y_test, y_preds)
    bacc.append(score)

    print(f"Seed {seed} - Test Balanced Accuracy: {score:.4f}")
    print(f"Best params: {search.best_params_}")

print(f"Mean Test Balanced Accuracy: {np.mean(bacc):.4f}")
print(f"Std Test Balanced Accuracy: {np.std(bacc):.4f}")

Seed 0 - Test Balanced Accuracy: 0.6767
Best params: {'clf__C': 1.0, 'rff__gamma': 0.1, 'rff__n_components': 500}
Seed 1 - Test Balanced Accuracy: 0.6787
Best params: {'clf__C': 0.1, 'rff__gamma': 0.1, 'rff__n_components': 2000}
Seed 2 - Test Balanced Accuracy: 0.6399
Best params: {'clf__C': 0.1, 'rff__gamma': 0.1, 'rff__n_components': 2000}
Seed 3 - Test Balanced Accuracy: 0.6455
Best params: {'clf__C': 0.1, 'rff__gamma': 0.1, 'rff__n_components': 1000}
Seed 4 - Test Balanced Accuracy: 0.6616
Best params: {'clf__C': 0.1, 'rff__gamma': 0.1, 'rff__n_components': 2000}
Seed 5 - Test Balanced Accuracy: 0.6787
Best params: {'clf__C': 0.1, 'rff__gamma': 0.1, 'rff__n_components': 500}
Seed 6 - Test Balanced Accuracy: 0.6567
Best params: {'clf__C': 0.1, 'rff__gamma': 0.1, 'rff__n_components': 500}
Seed 7 - Test Balanced Accuracy: 0.7093
Best params: {'clf__C': 1.0, 'rff__gamma': 0.1, 'rff__n_components': 2000}
Seed 8 - Test Balanced Accuracy: 0.6468
Best params: {'clf__C': 0.1, 'rff__gamma': 

In [ ]:
print("\n" + "="*50)
print("PHIÊN BẢN 2: ENSEMBLE (mạnh hơn nhưng chậm hơn)")
print("="*50)

# ========== PHIÊN BẢN 2: ENSEMBLE ==========
bacc_ensemble = []

for seed in range(30):
    pack = generate_data(seed=seed, use_scaler=False)
    X_train, y_train = pack["train"]
    X_test, y_test = pack["test"]

    tscv = TimeSeriesSplit(n_splits=3)

    # Tạo ensemble với nhiều seeds và gamma values
    members = []
    for rff_seed in [0, 1, 2]:
        for gamma in [0.1, 0.5, 1.0]:
            members.append((
                f"rff_{gamma}_{rff_seed}",
                Pipeline([
                    ("scaler", StandardScaler()),
                    ("rff", RBFSampler(gamma=gamma, n_components=1000, random_state=rff_seed)),
                    ("clf", LogisticRegression(max_iter=2000, random_state=42))
                ])
            ))

    # Ensemble classifier
    ensemble = VotingClassifier(estimators=members, voting="soft", n_jobs=-1)

    # Param grid cho ensemble (ít tham số hơn để tránh quá chậm)
    param_grid_ensemble = {
        # Tune C cho tất cả logistic regression trong ensemble
        "rff_0.1_0__clf__C": [0.1, 1.0, 10.0],
        "rff_0.5_1__clf__C": [0.1, 1.0, 10.0],
    }

    # Với GridSearch (có thể rất chậm)
    search_ensemble = GridSearchCV(
        estimator=ensemble,
        param_grid=param_grid_ensemble,
        cv=tscv,
        scoring=scorer,
        n_jobs=1,  # Giảm n_jobs để tránh quá tải
        refit=True
    )

    search_ensemble.fit(X_train, y_train)

    y_preds_ensemble = search_ensemble.predict(X_test)
    score_ensemble = balanced_accuracy_score(y_test, y_preds_ensemble)
    bacc_ensemble.append(score_ensemble)

    print(f"Seed {seed} - Ensemble Test Balanced Accuracy: {score_ensemble:.4f}")

print(f"Mean Ensemble Test Balanced Accuracy: {np.mean(bacc_ensemble):.4f}")
print(f"Std Ensemble Test Balanced Accuracy: {np.std(bacc_ensemble):.4f}")

# ========== SO SÁNH KẾT QUẢ ==========
print("\n" + "="*50)
print("TỔNG KẾT")
print("="*50)
print(f"RFF Non-ensemble: {np.mean(bacc):.4f} ± {np.std(bacc):.4f}")
print(f"RFF Ensemble:     {np.mean(bacc_ensemble):.4f} ± {np.std(bacc_ensemble):.4f}")


PHIÊN BẢN 2: ENSEMBLE (mạnh hơn nhưng chậm hơn)
Seed 0 - Ensemble Test Balanced Accuracy: 0.6818
Seed 1 - Ensemble Test Balanced Accuracy: 0.6188
Seed 2 - Ensemble Test Balanced Accuracy: 0.6147
Seed 3 - Ensemble Test Balanced Accuracy: 0.6273
Seed 4 - Ensemble Test Balanced Accuracy: 0.6043
Seed 5 - Ensemble Test Balanced Accuracy: 0.6341
Seed 6 - Ensemble Test Balanced Accuracy: 0.6318
Seed 7 - Ensemble Test Balanced Accuracy: 0.6643
Seed 8 - Ensemble Test Balanced Accuracy: 0.6704
Seed 9 - Ensemble Test Balanced Accuracy: 0.6078
Seed 10 - Ensemble Test Balanced Accuracy: 0.6791
Seed 11 - Ensemble Test Balanced Accuracy: 0.6564
Seed 12 - Ensemble Test Balanced Accuracy: 0.6437
Seed 13 - Ensemble Test Balanced Accuracy: 0.6154
Seed 14 - Ensemble Test Balanced Accuracy: 0.6672
Seed 15 - Ensemble Test Balanced Accuracy: 0.6578
Seed 16 - Ensemble Test Balanced Accuracy: 0.6191
Seed 17 - Ensemble Test Balanced Accuracy: 0.6436
Seed 18 - Ensemble Test Balanced Accuracy: 0.6566
Seed 19 - E